# Hostile Testing Phase 4 - Admin, Testing, Files Modules

## Purpose
Test admin, testing, and files modules - these are critical for security

## Goal
Add ~15 more functions to reach 4-5% coverage

In [ ]:
import pandas as pd
import tempfile
from pathlib import Path
import shutil

test_results = {
    'function': [],
    'module': [],
    'test_category': [],
    'passed': [],
    'error_message': [],
    'severity': []
}

def record_test(function, module, test_category, passed, error_message="", severity="medium"):
    test_results['function'].append(function)
    test_results['module'].append(module)
    test_results['test_category'].append(test_category)
    test_results['passed'].append(passed)
    test_results['error_message'].append(error_message)
    test_results['severity'].append(severity)

def hostile_test(func, test_name, *args, **kwargs):
    try:
        result = func(*args, **kwargs)
        return (True, result, "")
    except Exception as e:
        return (False, None, str(e))

print("Phase 4 test harness loaded")

## Section 1: Admin Module - Profile Management

In [ ]:
from siege_utilities import (
    get_default_profile_location, set_profile_location,
    get_profile_location, validate_profile_location
)

# Test get_default_profile_location
success, result, error = hostile_test(get_default_profile_location, "get_default")
record_test("get_default_profile_location", "admin.profile_manager", "basic_call", success, error, "low")
print(f"get_default_profile_location: {'✅' if success else '❌'}")

# Test set_profile_location with hostile paths
hostile_locations = [
    None, "", "../../../etc/passwd", "/dev/null", "location; rm -rf /",
    "A" * 10000, "path\x00injection", "~/.ssh/id_rsa", "/etc/shadow",
    "file:///etc/passwd", "\\\\network\\share", "C:\\Windows\\System32"
]

for loc in hostile_locations:
    success, result, error = hostile_test(set_profile_location, "hostile_loc", loc)
    record_test("set_profile_location", "admin.profile_manager", "hostile_locations",
                success, error,
                "critical" if "ssh" in str(loc) or "shadow" in str(loc) else "high" if "../" in str(loc) else "medium")

# Test validate_profile_location
for loc in hostile_locations:
    success, result, error = hostile_test(validate_profile_location, "validate_loc", loc)
    record_test("validate_profile_location", "admin.profile_manager", "hostile_locations",
                success, error, "high")

print(f"Completed {len([r for r in test_results['module'] if 'admin' in r])} admin tests")

## Section 2: Files Module - Path Operations

In [ ]:
from siege_utilities import (
    ensure_path_exists, get_file_extension, normalize_path,
    get_relative_path, create_backup_path, is_hidden_file
)

test_dir = Path(tempfile.mkdtemp())
try:
    # Test normalize_path with hostile paths
    hostile_paths = [
        None, "", "../../../etc/passwd", "/dev/null", "path; rm -rf /",
        "A" * 10000, "path\x00null", "~/../../../etc/shadow",
        "./././../../../etc/hosts", "C:\\..\\..\\Windows\\System32"
    ]
    
    for path in hostile_paths:
        success, result, error = hostile_test(normalize_path, "hostile_norm", path)
        record_test("normalize_path", "files.paths", "hostile_paths",
                    success, error,
                    "critical" if "etc" in str(path) else "high" if "../" in str(path) else "medium")
    
    # Test get_file_extension
    hostile_filenames = [
        None, "", "file; rm -rf /", "../../../etc/passwd",
        "file\x00.txt", "<script>.js", "'; DROP TABLE files; --.py",
        "A" * 10000 + ".txt"
    ]
    
    for fname in hostile_filenames:
        success, result, error = hostile_test(get_file_extension, "hostile_ext", fname)
        record_test("get_file_extension", "files.paths", "hostile_filenames",
                    success, error, "medium")
    
    # Test is_hidden_file
    for fname in hostile_filenames:
        success, result, error = hostile_test(is_hidden_file, "hostile_hidden", fname)
        record_test("is_hidden_file", "files.paths", "hostile_filenames",
                    success, error, "medium")
    
    # Test create_backup_path
    for path in hostile_paths:
        success, result, error = hostile_test(create_backup_path, "hostile_backup", path)
        record_test("create_backup_path", "files.paths", "hostile_paths",
                    success, error,
                    "high" if "../" in str(path) else "medium")
    
finally:
    shutil.rmtree(test_dir, ignore_errors=True)

print(f"Completed {len([r for r in test_results['module'] if 'files' in r])} files tests")

## Section 3: Files Module - File Operations

In [ ]:
from siege_utilities import (
    file_exists, touch_file, get_file_size,
    check_if_file_exists_at_path
)

test_dir = Path(tempfile.mkdtemp())
try:
    # Test file_exists with hostile paths
    hostile_paths = [
        None, "", "../../../etc/passwd", "/dev/null", "file; rm -rf /",
        "A" * 10000, "file\x00.txt", "~/../../../etc/shadow"
    ]
    
    for path in hostile_paths:
        success, result, error = hostile_test(file_exists, "hostile_exists", path)
        record_test("file_exists", "files.operations", "hostile_paths",
                    success, error,
                    "critical" if "etc" in str(path) else "high" if "../" in str(path) else "medium")
    
    # Test check_if_file_exists_at_path
    for path in hostile_paths:
        success, result, error = hostile_test(check_if_file_exists_at_path, "hostile_check", path)
        record_test("check_if_file_exists_at_path", "files.operations", "hostile_paths",
                    success, error,
                    "critical" if "etc" in str(path) else "high" if "../" in str(path) else "medium")
    
    # Test touch_file (DON'T actually create files with hostile names)
    # Only test within our temp directory
    safe_test_paths = [
        test_dir / "test.txt",
        test_dir / "file with spaces.txt",
        test_dir / "中文文件.txt"
    ]
    
    for path in safe_test_paths:
        success, result, error = hostile_test(touch_file, "safe_touch", str(path))
        record_test("touch_file", "files.operations", "safe_paths",
                    success, error, "low")
    
    # Test get_file_size with hostile paths (won't actually access them)
    for path in hostile_paths:
        success, result, error = hostile_test(get_file_size, "hostile_size", path)
        record_test("get_file_size", "files.operations", "hostile_paths",
                    success, error,
                    "critical" if "etc" in str(path) else "high" if "../" in str(path) else "medium")
    
finally:
    shutil.rmtree(test_dir, ignore_errors=True)

print(f"Completed file operations tests")

## Section 4: Files Module - Hashing

In [ ]:
from siege_utilities import (
    calculate_file_hash, get_file_hash, get_quick_file_signature
)

test_dir = Path(tempfile.mkdtemp())
try:
    # Create a legitimate test file
    test_file = test_dir / "test.txt"
    test_file.write_text("test content")
    
    # Test with hostile paths (should fail safely)
    hostile_paths = [
        None, "", "../../../etc/passwd", "/dev/null",
        "A" * 10000, "file\x00.txt"
    ]
    
    for path in hostile_paths:
        success, result, error = hostile_test(calculate_file_hash, "hostile_hash", path)
        record_test("calculate_file_hash", "files.hashing", "hostile_paths",
                    success, error,
                    "critical" if "etc" in str(path) else "high" if "../" in str(path) else "medium")
    
    # Test get_file_hash
    for path in hostile_paths:
        success, result, error = hostile_test(get_file_hash, "hostile_get_hash", path)
        record_test("get_file_hash", "files.hashing", "hostile_paths",
                    success, error,
                    "critical" if "etc" in str(path) else "high" if "../" in str(path) else "medium")
    
    # Test get_quick_file_signature
    for path in hostile_paths:
        success, result, error = hostile_test(get_quick_file_signature, "hostile_sig", path)
        record_test("get_quick_file_signature", "files.hashing", "hostile_paths",
                    success, error,
                    "critical" if "etc" in str(path) else "high" if "../" in str(path) else "medium")
    
finally:
    shutil.rmtree(test_dir, ignore_errors=True)

print(f"Completed hashing tests")

## Section 5: Testing Module

In [ ]:
from siege_utilities import get_system_info

# Test get_system_info (no args)
success, result, error = hostile_test(get_system_info, "get_sys_info")
record_test("get_system_info", "testing.environment", "basic_call",
            success, error, "low")

print(f"get_system_info: {'✅' if success else '❌'}")
print(f"Completed testing module tests")

## Section 6: Generate Phase 4 Results

In [ ]:
df = pd.DataFrame(test_results)

print("\n" + "="*80)
print("PHASE 4 HOSTILE TESTING SUMMARY")
print("="*80)
print(f"\nTests run: {len(df)}")
print(f"Passed: {df['passed'].sum()}")
print(f"Failed: {(~df['passed']).sum()}")
print(f"Pass rate: {df['passed'].sum() / len(df) * 100:.1f}%")

unique = df['function'].nunique()
print(f"\nUnique functions: {unique}")
print(f"Phase 4 coverage: {unique}/751 = {unique/751*100:.1f}%")

print("\n" + "="*80)
print("RESULTS BY MODULE")
print("="*80)
summary = df.groupby('module').agg({'passed': ['sum', 'count']})
summary.columns = ['passed', 'total']
summary['pass_rate'] = (summary['passed'] / summary['total'] * 100).round(1)
print(summary)

failures = df[~df['passed']]
if len(failures) > 0:
    print("\n" + "="*80)
    print("FAILURES BY SEVERITY")
    print("="*80)
    print(failures.groupby('severity').size())
    
    critical_high = failures[failures['severity'].isin(['critical', 'high'])]
    if len(critical_high) > 0:
        print("\n" + "="*80)
        print("CRITICAL/HIGH FAILURES (First 10)")
        print("="*80)
        for idx, row in critical_high.head(10).iterrows():
            print(f"\n{row['function']} ({row['severity']})")
            print(f"  Module: {row['module']}")
            print(f"  Error: {row['error_message'][:100]}")
else:
    print("\n✅ NO FAILURES")

df.to_csv("hostile_testing_phase4_results.csv", index=False)
print(f"\nSaved: hostile_testing_phase4_results.csv")

print("\n" + "="*80)
print("FUNCTIONS TESTED")
print("="*80)
for func in df['function'].unique():
    tests = df[df['function'] == func]
    passed = tests['passed'].sum()
    total = len(tests)
    print(f"{func}: {passed}/{total} ({passed/total*100:.0f}%)")

print("\n" + "="*80)
print("CUMULATIVE COVERAGE (Phases 1-4)")
print("="*80)
total_unique = 19 + unique  # Previous phases had 19 unique
print(f"Total unique functions: ~{total_unique}")
print(f"Cumulative coverage: ~{total_unique}/751 = {total_unique/751*100:.1f}%")